In [0]:
# Cell 1: imports, settings and city config
import os
import json
import requests
import yaml
from datetime import date, datetime, timezone

CATALOG = "city_delta"
USER_AGENT = "CityDelta/1.0 (portfolio project)"
BRONZE_TABLE = f"{CATALOG}.bronze.weather_raw"
VOLUME_ROOT = f"/Volumes/{CATALOG}/bronze/raw/weather"
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"

with open("../config/cities.yaml") as f:
    cities = yaml.safe_load(f)["cities"]

today = date.today()
ingest_ts = datetime.now(timezone.utc)
print([c["key"] for c in cities])

In [0]:
# Cell 2: fetch the hourly forecast per city, save raw JSON, collect rows
HOURLY_VARS = "temperature_2m,precipitation_probability,precipitation,weather_code,wind_speed_10m"

rows = []
for city in cities:
    params = {
        "latitude": city["lat"],
        "longitude": city["lon"],
        "hourly": HOURLY_VARS,
        "timezone": "Europe/Amsterdam",
        "forecast_days": 7,
    }
    resp = requests.get(WEATHER_URL, params=params,
                        headers={"User-Agent": USER_AGENT}, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    # Keep the raw response as a file
    out_dir = f"{VOLUME_ROOT}/{city['key']}"
    os.makedirs(out_dir, exist_ok=True)
    raw_file = f"{out_dir}/{today.isoformat()}.json"
    with open(raw_file, "w", encoding="utf-8") as f:
        json.dump(data, f)

    rows.append({
        "city_key": city["key"],
        "ingest_ts": ingest_ts,
        "ingest_date": today,
        "request_url": resp.url,
        "raw_file": raw_file,
        "raw_json": json.dumps(data),
    })
    print(f"{city['key']}: {len(data['hourly']['time'])} hourly values")

In [0]:
# Cell 3: write to the bronze Delta table (idempotent per day) and check
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DateType

schema = StructType([
    StructField("city_key",    StringType(),    False),
    StructField("ingest_ts",   TimestampType(), False),
    StructField("ingest_date", DateType(),      False),
    StructField("request_url", StringType(),    True),
    StructField("raw_file",    StringType(),    True),
    StructField("raw_json",    StringType(),    False),
])

df = spark.createDataFrame(rows, schema=schema)

if spark.catalog.tableExists(BRONZE_TABLE):
    spark.sql(f"DELETE FROM {BRONZE_TABLE} WHERE ingest_date = DATE'{today.isoformat()}'")

df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)

display(spark.sql(f"SELECT ingest_date, city_key, length(raw_json) AS json_chars FROM {BRONZE_TABLE}"))